# Обратный Эйлер и Рунге–Кутта 4

Моделируем ту же систему «масса — пружина — демпфер», что и в `modeling_n_simulation.ipynb`. Координата $x$ направлена вверх, состояние $y=(x,v)$, а уравнение движения имеет вид $m\ddot{x}+b\dot{x}+kx=-mg$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

MASS = 1.0
DAMPING = 0.5
STIFFNESS = 20.0
GRAVITY = 9.8


def dynamics(state):
    """Return derivatives of the state [position, velocity]."""
    position, velocity = state
    acceleration = -(DAMPING * velocity + STIFFNESS * position) / MASS - GRAVITY
    return np.array([velocity, acceleration])

## Обратный Эйлер

В неявном шаге $y_{n+1}=y_n+h f(y_{n+1})$ производные считаем в новом состоянии. Запишем шаг отдельно для положения и скорости.

$$
\begin{aligned}
x_{n+1} &= x_n + h v_{n+1}, \\
v_{n+1} &= v_n - h\left(\frac{b}{m}v_{n+1}+\frac{k}{m}x_{n+1}+g\right).
\end{aligned}
$$

Подставляем выражение для $x_{n+1}$ во второе уравнение.

$$
v_{n+1} = v_n - h\left(\frac{b}{m}v_{n+1}+\frac{k}{m}(x_n+h v_{n+1})+g\right).
$$

Раскрываем скобки и переносим все слагаемые с $v_{n+1}$ в левую часть.

$$
\left(1+\frac{hb}{m}+\frac{h^2k}{m}\right)v_{n+1}
= v_n-h\left(\frac{kx_n}{m}+g\right).
$$

Делим на коэффициент перед $v_{n+1}$.

$$
v_{n+1}=\frac{v_n-h(kx_n/m+g)}{1+hb/m+h^2k/m}.
$$

Затем вычисляем новое положение по формуле $x_{n+1}=x_n+h v_{n+1}$.

In [ ]:
def backward_euler_step(state, h):
    """Solve the implicit update for this linear system."""
    position, velocity = state
    # Substitute x_next = position + h * v_next into the velocity update.
    velocity_next = (velocity - h * (STIFFNESS * position / MASS + GRAVITY)) / (
        1 + h * DAMPING / MASS + h**2 * STIFFNESS / MASS
    )
    position_next = position + h * velocity_next
    return np.array([position_next, velocity_next])

## Рунге–Кутта 4

Состояние $y_n=(x_n,v_n)$ содержит положение и скорость, а функция $f(y)$ возвращает их производные.

$$
f(y)=
\begin{bmatrix}
v \\
-(bv+kx)/m-g
\end{bmatrix}.
$$

На шаге длины $h$ вычисляем четыре оценки производной. Первая берётся в начале шага, вторая и третья — в двух приближённых состояниях в середине шага, четвёртая — в приближённом состоянии в конце шага.

$$
\begin{aligned}
k_1 &= f(y_n), \\
k_2 &= f\left(y_n+\frac{h}{2}k_1\right), \\
k_3 &= f\left(y_n+\frac{h}{2}k_2\right), \\
k_4 &= f(y_n+h k_3).
\end{aligned}
$$

Умножаем взвешенное среднее этих оценок на $h$ и прибавляем к текущему состоянию.

$$
y_{n+1}=y_n+\frac{h}{6}\left(k_1+2k_2+2k_3+k_4\right).
$$

Каждая оценка $k_i$ содержит две компоненты, поэтому этот шаг обновляет и положение, и скорость.

In [ ]:
def rk4_step(state, h):
    """Take one classical fourth-order Runge–Kutta step."""
    k1 = dynamics(state)
    k2 = dynamics(state + h * k1 / 2)
    k3 = dynamics(state + h * k2 / 2)
    k4 = dynamics(state + h * k3)
    return state + h * (k1 + 2 * k2 + 2 * k3 + k4) / 6

## Сравнение результатов

Начинаем из состояния $x_0=0.5$ м, $v_0=0$ м/с и моделируем 10 с с шагом 0.01 с. Сравниваем положение и скорость на отдельных графиках.

In [ ]:
def simulate(step, initial_state, h, steps):
    time = h * np.arange(steps + 1)
    states = np.empty((steps + 1, 2))
    states[0] = initial_state
    for index in range(steps):
        states[index + 1] = step(states[index], h)
    return time, states

In [ ]:
initial_state = np.array([0.5, 0.0])
h = 0.01
steps = 1000

_, axes = plt.subplots(2, 1, sharex=True)
for name, method in (("Backward Euler", backward_euler_step), ("RK4", rk4_step)):
    time, states = simulate(method, initial_state, h, steps)
    axes[0].plot(time, states[:, 0], label=name)
    axes[1].plot(time, states[:, 1], label=name)

axes[0].set_ylabel("Position [m]")
axes[1].set_ylabel("Velocity [m/s]")
axes[1].set_xlabel("Time [s]")
for axis in axes:
    axis.grid()
    axis.legend()
plt.tight_layout()
plt.show()